## OSM Walk Network

`NYCPathways.ipynb` built a routing network from NYC's own Street Centerline dataset plus a custom skeleton extracted from park polygons. This notebook pulls the equivalent network from OpenStreetMap via `osmnx` instead, as in `04_networks.ipynb`, to compare against (or replace) that source.

This notebook only imports the full walk network for both study neighborhoods. Splitting it into roads vs. dedicated paths (sidewalks, park paths, etc.) is a separate next step.

In [ ]:
import geopandas as gpd
import pandas as pd
import osmnx as ox
import matplotlib.pyplot as plt
from cdptools import utils

utils.set_axis_off()

Reuse the same neighborhood boundaries as the other notebooks in this project (`NYCPathways.ipynb`, `Neighborhood_Population_Distribution.ipynb`).

In [ ]:
neighborhoods = {
    "bedstuy": "Bedford-Stuyvesant",
    "jacksonheights": "Jackson Heights",
}

boundaries = {
    key: gpd.read_file(f"Data/boundary_{key}.geojson").to_crs(4326)
    for key in neighborhoods
}

Pull the OSM `walk` network for each boundary — this is everything a pedestrian can use: streets with sidewalks, dedicated footways, park paths, pedestrian plazas. We split this into roads vs. dedicated paths afterward, so we want everything walkable in one pass here, not a pre-filtered subset.

`retain_all=True` keeps disconnected pieces (e.g. a park path system not yet linked to the street grid) instead of silently dropping everything but the single largest connected component. `truncate_by_edge=True` keeps edges that cross the boundary rather than clipping them into artificial dead ends right at the neighborhood line.

In [ ]:
graphs = {
    key: ox.graph_from_polygon(
        boundary.union_all(),
        network_type="walk",
        retain_all=True,
        truncate_by_edge=True,
    )
    for key, boundary in boundaries.items()
}

Convert each graph to node/edge geodataframes and tag both with their neighborhood, matching the pattern used for `pluto_gdf` and `walk_network` elsewhere in this project.

In [ ]:
nodes, edges = {}, {}
for key, G in graphs.items():
    n, e = ox.graph_to_gdfs(G)
    n["neighborhood"] = key
    e["neighborhood"] = key
    nodes[key] = n
    edges[key] = e

all_nodes = gpd.GeoDataFrame(pd.concat(nodes.values()))
all_edges = gpd.GeoDataFrame(pd.concat(edges.values()))

Quick visual check, same layout as the network plots elsewhere in the project.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, key in zip(axes, boundaries):
    edges[key].plot(ax=ax, color="black", linewidth=0.3)
    nodes[key].plot(ax=ax, color="orange", markersize=1)
    boundaries[key].boundary.plot(ax=ax, color="blue", linewidth=1)
    ax.set_title(neighborhoods[key])

Sanity-check counts, plus the `highway` tag distribution — this is the field the next step (splitting into roads vs. paths) will key off of.

In [ ]:
for key in neighborhoods:
    print(f"{neighborhoods[key]}: {len(nodes[key])} nodes, {len(edges[key])} edges")

all_edges["highway"].explode().value_counts()